In [1]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import GroupShuffleSplit

from tqdm.auto import tqdm

import wandb
import warnings
warnings.filterwarnings("ignore")


In [2]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train_df.shape)
print(test_df.shape)

def normalize_prompt(text):

    text = text.lower()

    text = text.replace("based on the given context", "")
    text = text.replace("carefully", "")
    text = text.replace("among the listed options", "")
    text = text.replace("from the following choices", "")

    return " ".join(text.split())


train_df["group"] = train_df["prompt"].apply(normalize_prompt)
test_df["group"] = test_df["prompt"].apply(normalize_prompt)

train_df.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer,group
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,pick the best possible answer: what is martin ...
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,what is accelerator-based light-ion fusion?
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,determine the correct option: what is the term...
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,select the most accurate option: what is marti...
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,identify the correct statement: what is the co...


In [3]:
def make_pairwise(df, train=True):

    rows = []

    for _, row in df.iterrows():

        for option in ["A", "B", "C", "D", "E"]:

            rows.append({

                "id": row["id"],

                "group": row["group"],

                "text":
                f"Question: {row['prompt']} [SEP] Option: {row[option]}",

                "option": option,

                "label":
                int(option == row["answer"]) if train else -1

            })

    return pd.DataFrame(rows)


pair_train = make_pairwise(train_df, train=True)
pair_test = make_pairwise(test_df, train=False)

print(pair_train.shape)
print(pair_test.shape)

pair_train.head()

(10000, 5)
(2500, 5)


,id,group,text,option,label
0,1,pick the best possible answer: what is martin ...,Question: Pick the best possible answer: What ...,A,0
1,1,pick the best possible answer: what is martin ...,Question: Pick the best possible answer: What ...,B,1
2,1,pick the best possible answer: what is martin ...,Question: Pick the best possible answer: What ...,C,0
3,1,pick the best possible answer: what is martin ...,Question: Pick the best possible answer: What ...,D,0
4,1,pick the best possible answer: what is martin ...,Question: Pick the best possible answer: What ...,E,0


In [4]:
MODEL_NAME = "bert-base-uncased"
MAX_LEN    = 256
BATCH_SIZE = 16
EPOCHS     = 4
LR         = 2e-5

wandb.login(key="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")

run = wandb.init(
    project="23f3001092-t22026",
    name="bert-base-uncased-mcq",
    config={
        "model":      MODEL_NAME,
        "max_len":    MAX_LEN,
        "batch_size": BATCH_SIZE,
        "epochs":     EPOCHS,
        "lr":         LR,
        "metric":     "MAP@3"
    }
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

groups = pair_train["group"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, valid_idx = next(
    gss.split(
        pair_train,
        pair_train["label"],
        groups
    )
)

train_pair = pair_train.iloc[train_idx].reset_index(drop=True)
valid_pair = pair_train.iloc[valid_idx].reset_index(drop=True)

print("Train pairs:", train_pair.shape)
print("Valid pairs:", valid_pair.shape)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train pairs: (7995, 5)
Valid pairs: (2005, 5)


In [5]:
class MCQDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_len=MAX_LEN):

        self.data      = dataframe
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        text = self.data.iloc[idx]["text"]

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            truncation=True,
            padding="max_length",
            return_attention_mask=True
        )

        item = {
            "input_ids": torch.tensor(
                encoding["input_ids"],
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                encoding["attention_mask"],
                dtype=torch.long
            )
        }

        if "label" in self.data.columns and self.data.iloc[idx]["label"] != -1:
            item["labels"] = torch.tensor(
                self.data.iloc[idx]["label"],
                dtype=torch.float
            )

        return item


In [6]:
train_dataset = MCQDataset(train_pair, tokenizer)
valid_dataset = MCQDataset(valid_pair, tokenizer)
test_dataset  = MCQDataset(pair_test,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=32,         shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32,         shuffle=False, num_workers=2, pin_memory=True)

print("Train batches:", len(train_loader))
print("Valid batches:", len(valid_loader))
print("Test  batches:", len(test_loader))


Train batches: 500
Valid batches: 63
Test  batches: 79


In [7]:
batch = next(iter(train_loader))

# print(batch["input_ids"].shape) //extra
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([16, 256])
torch.Size([16])


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

class BertRanker(nn.Module):

    def __init__(self):

        super().__init__()

        self.bert = AutoModel.from_pretrained("bert-base-uncased")

        self.dropout = nn.Dropout(0.2)

        self.fc = nn.Linear(
            self.bert.config.hidden_size,
            1
        )

    def forward(self, input_ids, attention_mask):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls = outputs.last_hidden_state[:, 0]

        cls = self.dropout(cls)

        score = self.fc(cls)

        return score.squeeze(-1)


model = BertRanker().to(device)

optimizer = AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01
)

criterion = nn.BCEWithLogitsLoss()

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print("Model loaded")
print(f"Total steps: {total_steps}  Warmup steps: {warmup_steps}")


device: cuda


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded
Total steps: 1500  Warmup steps: 150


In [9]:
# extra
batch = next(iter(train_loader))

with torch.no_grad():

    logits = model(
        batch["input_ids"].to(device),
        batch["attention_mask"].to(device)
    )

print(logits[:5])

print(torch.isnan(logits).any()) 

tensor([-0.2383, -0.0661, -0.0167, -0.3098, -0.0052], device='cuda:0')
tensor(False, device='cuda:0')


In [10]:
def map3(y_true, y_pred):
    score = 0
    for t, p in zip(y_true, y_pred):
        if   t == p[0]: score += 1.0
        elif t == p[1]: score += 0.5
        elif t == p[2]: score += 1/3
    return score / len(y_true)


def evaluate(model, loader, pair_df):
    model.eval()
    scores = []
    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            logits = model(ids, mask)
            probs  = torch.sigmoid(logits)
            scores.extend(probs.cpu().numpy())

    df = pair_df.copy()
    df["score"] = scores

    preds, truths = [], []
    for _, grp in df.groupby("id"):
        grp = grp.sort_values("score", ascending=False)
        preds.append(grp["option"].head(3).tolist())
        truths.append(grp[grp["label"] == 1]["option"].values[0])

    val_map3 = map3(truths, preds)
    val_acc  = float(np.mean([t == p[0] for t, p in zip(truths, preds)]))
    return val_map3, val_acc


best_map3 = 0.0

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for step, batch in enumerate(loop):

        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        loop.set_postfix(loss=loss.item())

        wandb.log({
            "train/step_loss": loss.item(),
            "train/lr":        scheduler.get_last_lr()[0]
        })

    avg_loss = total_loss / len(train_loader)

    val_map3, val_acc = evaluate(model, valid_loader, valid_pair)

    print(f"Epoch {epoch+1}  Loss: {avg_loss:.4f}  Val MAP@3: {val_map3:.4f}  Val Acc: {val_acc:.4f}")

    wandb.log({
        "epoch":           epoch + 1,
        "train/epoch_loss": avg_loss,
        "val/map3":         val_map3,
        "val/accuracy":     val_acc
    })

    if val_map3 > best_map3:
        best_map3 = val_map3
        torch.save(model.state_dict(), "best_bert.pt")
        print(f"  Best model saved (MAP@3={best_map3:.4f})")

print(f"\nBest Val MAP@3: {best_map3:.4f}")
wandb.log({"best_val_map3": best_map3})


Epoch 1/3:   0%|          | 0/500 [00:00<?, ?it/s]

Epoch 1  Loss: 0.4816  Val MAP@3: 0.9065  Val Acc: 0.8603
  Best model saved (MAP@3=0.9065)


Epoch 2/3:   0%|          | 0/500 [00:00<?, ?it/s]

Epoch 2  Loss: 0.2465  Val MAP@3: 0.9626  Val Acc: 0.9476
  Best model saved (MAP@3=0.9626)


Epoch 3/3:   0%|          | 0/500 [00:00<?, ?it/s]

Epoch 3  Loss: 0.1436  Val MAP@3: 0.9676  Val Acc: 0.9576
  Best model saved (MAP@3=0.9676)

Best Val MAP@3: 0.9676


In [11]:
model.load_state_dict(torch.load("best_bert.pt", map_location=device))
model.eval()

valid_scores = []

with torch.no_grad():

    for batch in tqdm(valid_loader):

        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits = model(input_ids, attention_mask)
        scores = torch.sigmoid(logits)

        valid_scores.extend(scores.cpu().numpy())

valid_scores = np.array(valid_scores)
print("Valid scores sample:", valid_scores[:5])


  0%|          | 0/63 [00:00<?, ?it/s]

Valid scores sample: [0.00134945 0.00143681 0.99761355 0.00164553 0.00253961]


In [12]:
valid_df = valid_pair.copy()
valid_df["score"] = valid_scores

predictions  = []
true_answers = []

for _, group in valid_df.groupby("id"):

    group = group.sort_values("score", ascending=False)

    predictions.append(group["option"].head(3).tolist())

    true_answers.append(
        group[group["label"] == 1]["option"].values[0]
    )

acc      = np.mean([t == p[0] for t, p in zip(true_answers, predictions)])
val_map3 = map3(true_answers, predictions)

print("Validation MAP@3 :", round(val_map3, 4))
print("Validation Accuracy :", round(acc, 4))

wandb.log({
    "final/val_map3":    val_map3,
    "final/val_accuracy": acc
})


Validation MAP@3 : 0.9676
Validation Accuracy : 0.9576


In [13]:
model.eval()

test_scores = []

with torch.no_grad():

    for batch in tqdm(test_loader):

        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits = model(input_ids, attention_mask)
        scores = torch.sigmoid(logits)

        test_scores.extend(scores.cpu().numpy())

test_scores = np.array(test_scores)
print("Test scores computed:", test_scores.shape)


  0%|          | 0/79 [00:00<?, ?it/s]

Test scores computed: (2500,)


In [14]:
test_result = pair_test.copy()
test_result["score"] = test_scores

predictions = []

for _, group in test_result.groupby("id"):

    group = group.sort_values("score", ascending=False)

    predictions.append(
        " ".join(group["option"].head(3).tolist())
    )

submission = pd.DataFrame({
    "ID":         test_df["id"],
    "Prediction": predictions
})

submission.to_csv("submission.csv", index=False)

artifact = wandb.Artifact("bert-submission", type="predictions")
artifact.add_file("submission.csv")
run.log_artifact(artifact)

run.finish()

print(submission.head())


best_val_map3,▁
epoch,▁▅█
final/val_accuracy,▁
final/val_map3,▁
train/epoch_loss,█▃▁
train/lr,▂████████▇▆▆▆▆▅▅▅▅▄▄▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁
train/step_loss,▅▇▅▇▆▇▆▅██▅▅▆▅▅▆▄▆▅▂▁▄▃▁▃▂▃▁▂▁▄▄▁▃▂▅▁▆▁▂
val/accuracy,▁▇█
val/map3,▁▇█
best_val_map3,0.96758
epoch,3


   ID Prediction
0   1      A C B
1   2      B C A
2   3      B E D
3   4      E D C
4   5      C A D


In [15]:
print(submission.shape)
print(submission["Prediction"].head())

(500, 2)
0    A C B
1    B C A
2    B E D
3    E D C
4    C A D
Name: Prediction, dtype: object
